<a href="https://colab.research.google.com/github/ladparag100/AI-Projects/blob/main/2-OpsPilot-AI-Incident-Response/notebooks/OpsPilot_AI_Incident_Response.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# **OpsPilot-AI-Incident-Response**

## Project Overview

OpsPilot is an autonomous AI-powered incident response system designed to act as the intelligent first responder for server infrastructure incidents. It autonomously investigates infrastructure anomalies, performs root cause analysis, and executes appropriate remediation strategies or escalates to human engineers when necessary.

---

## Core Architecture & Features

**The AI agent operates through three intelligent layers:**

1. **Diagnostic Layer - Investigation:** Performs comprehensive server health analysis by monitoring CPU and memory metrics, analyzing system logs, and detecting anomalies when incidents are reported.

2. **Remediation Layer - Automated Action:** Executes automatic service restarts when resource utilization reaches critical thresholds (CPU or Memory >90%), enabling self-healing infrastructure.

3. **Escalation Layer - Intelligent Routing:** Identifies complex issues such as external dependency failures and routes them to human engineers for specialized investigation and resolution.

---

## System Requirements

- OpenAI Python library (openai)
- Valid OpenAI API key configured as an environment variable or notebook secret
- Access to GPT-4o-mini or compatible model

colab_secret.jpg

In [ ]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [ ]:
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# System Initialization

==========================================
## Step 1: Diagnostic & Remediation Tools
==========================================

In [ ]:
# Diagnostic Tool 1: Real-time Health Monitoring
def get_server_health(server_id: str) -> str:
    """Queries and returns current CPU and Memory utilization metrics for a specified server."""
    print(f"-> TOOL: Checking health for {server_id}...")

    metrics = {
        # Server State 1: High CPU Utilization - Requires Immediate Restart
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Server State 2: Optimal Performance - No Action Needed
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Server State 3: Critical Memory Pressure - Memory Leak Detected
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Server State 4: Healthy Metrics with Dependency Failure - Requires Escalation
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Server State 5: Baseline Healthy State - All Systems Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

    result = metrics.get(server_id, {"error": "Server not found. Check the ID."})
    return json.dumps(result)


In [ ]:
# Diagnostic Tool 2: Log Analysis & Error Detection
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Retrieves and returns the most recent log entries from a server for error pattern analysis."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")

    # Simulated log database representing different failure patterns
    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

    # Default logs if server not found in specific list
    default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

    logs = log_database.get(server_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

---
### Remediation & Escalation Tools
---


In [ ]:
# Remediation Tool: Automated Service Recovery
def restart_service(server_id: str) -> str:
    """
    Executes an automated service restart on the target server.
    Triggered when resource utilization (CPU or Memory) exceeds critical thresholds.
    Enables rapid recovery from resource exhaustion incidents.
    """
    print(f"-> TOOL: Restarting service for {server_id}...")
    return json.dumps({"status": "success", "message": "Server restarted successfully"})



In [ ]:
# Escalation Tool: Human-in-the-Loop Incident Routing
def escalate_to_engineer(summary: str) -> str:
    """
    Routes incidents to human engineers for manual investigation and resolution.
    Used when issues involve complex root causes or external dependency failures
    that cannot be resolved through automated remediation.
    Creates support ticket with incident summary for engineer review.
    """
    print(f"-> TOOL: Escalating to human...")
    return json.dumps({"status": "success", "message": "Ticket created successfully"})



In [ ]:
# Tool Registry - Maps tool names to their implementations for agent execution
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer,
}

==========================================
## Step 2: AI Agent Tool Definitions
==========================================

In [ ]:
# OpenAI Function Calling Schema
# Defines the complete set of tools available to the AI agent for incident response
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Retrieves real-time CPU and memory utilization metrics for infrastructure health assessment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The unique identifier of the server (e.g., 'payment-server-01')"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Analyzes recent system logs to identify error patterns, warnings, and critical events that indicate root causes.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The unique identifier of the server."},
                    "lines": {"type": "integer", "description": "Number of recent log entries to retrieve."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "Initiates an automated service restart when high resource utilization is detected, enabling self-healing of resource exhaustion issues.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The unique identifier of the server requiring restart (e.g., 'payment-server-01')"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Routes complex incidents to human engineers when automated remediation is insufficient or when external dependency failures are detected.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string", "description": "Detailed summary of the incident including diagnostics and recommended investigation areas."}
                },
                "required": ["summary"]
            }
        }
    }
]

 ==========================================
## Step 3: Incident Response Agent Loop
 ==========================================

**Agent Decision-Making Process:**

1. **Incident Reception:** Accepts incident reports from operations teams or monitoring systems
2. **Diagnostic Analysis:** Executes health checks and log analysis to establish root cause
3. **Intelligent Decision Making:**
   - Resource Exhaustion Detected (CPU/Memory >90%) → Execute automated restart
   - External Dependency Issues (Connection/Availability Errors) → Escalate to engineer
   - System Operating Normally → Report status and close incident
4. **Resolution Delivery:** Provides incident status and resolution details back to operations

In [ ]:
def run_it_agent(user_issue: str):
    """Executes the complete incident response workflow from detection through resolution."""
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are an AI-powered Level 1 Incident Response Agent. Your role is to investigate infrastructure incidents comprehensively. "
                                      "When CPU or Memory utilization exceeds 90%, execute immediate service restart. When logs indicate critical infrastructure failures (such as connection refused or dependency unavailability) that require specialized expertise, escalate the incident to senior engineers. Always conduct thorough analysis before taking action and ensure complete incident context is provided."},
        {"role": "user", "content": user_issue}
    ]

    while True:
        print("\n[AI Agent Analyzing Incident...]")
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                tool_call_id = tool_call.id
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Retrieve and execute the appropriate tool
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)

                if function_to_call:
                    # Execute diagnostic or remediation action
                    tool_output = function_to_call(**func_args)

                    # Return tool results to the AI agent for further processing
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call_id,
                        "name": func_name,
                        "content": tool_output
                    })

        else:
            # Agent has completed analysis and decision making
            print(f"\n[INCIDENT RESOLUTION]: {response_msg.content}")
            break

 ==========================================
## Step 4: Operational Incident Examples
 ==========================================

The following examples demonstrate the agent's incident response capabilities across various real-world infrastructure scenarios.

In [ ]:
# Example 1: Critical CPU Exhaustion - Automated Recovery
# Demonstrates: Health check detection → Service restart execution
# Scenario: Payment processing service experiencing 98% CPU utilization
run_it_agent("The payment-server-01 is experiencing severe performance degradation with connection timeouts.")


--- New Incident: The payment-server-01 is experiencing severe performance degradation with connection timeouts. ---

[AI Agent Analyzing Incident...]
-> TOOL: Checking health for payment-server-01...

[AI Agent Analyzing Incident...]
-> TOOL: Restarting service for payment-server-01...

[AI Agent Analyzing Incident...]
-> TOOL: Fetching last 50 log lines for payment-server-01...

[AI Agent Analyzing Incident...]
-> TOOL: Escalating to human...

[AI Agent Analyzing Incident...]

[INCIDENT RESOLUTION]: I have initiated an automated service restart for payment-server-01, which was operating at critical CPU utilization (98%). The system also detected indicators of process saturation and connection pool exhaustion from the logs. Due to the severity and complexity of the underlying issues, I have escalated this incident to the engineering team for comprehensive root cause analysis and permanent remediation.


In [ ]:
# Example 2: False Positive Detection - No Action Required
# Demonstrates: Operational health verification
# Scenario: Database node operating within normal parameters
run_it_agent("Verify db-node-02 status and investigate reported anomalies.")


--- New Incident: Verify db-node-02 status and investigate reported anomalies. ---

[AI Agent Analyzing Incident...]
-> TOOL: Checking health for db-node-02...
-> TOOL: Fetching last 50 log lines for db-node-02...

[AI Agent Analyzing Incident...]

[INCIDENT RESOLUTION]: Health assessment for db-node-02 indicates optimal operational status. CPU utilization is at 12% and memory utilization is at 60%, both well within acceptable parameters. System logs show normal operation with completed backups and active replication synchronization. No critical errors or anomalies detected. The system is operating as expected. If additional concerns arise, please provide specific symptoms or error indicators for further investigation.


In [ ]:
# Example 3: Memory Leak Incident - Automated Restart Execution
# Demonstrates: Memory pressure detection → Service restart
# Scenario: Authentication service experiencing memory exhaustion with OutOfMemoryError
run_it_agent("Authentication service is rejecting user login requests with service unavailable errors.")

print("\n" + "="*50 + "\n")


--- New Incident: Authentication service is rejecting user login requests with service unavailable errors. ---

[AI Agent Analyzing Incident...]
-> TOOL: Checking health for auth-service-03...
-> TOOL: Fetching last 50 log lines for auth-service-03...

[AI Agent Analyzing Incident...]
-> TOOL: Restarting service for auth-service-03...

[AI Agent Analyzing Incident...]

[INCIDENT RESOLUTION]: I have successfully executed an automated service restart for auth-service-03. The incident was caused by critical memory exhaustion with 95% utilization and application-level OutOfMemoryError conditions. The restart action has been completed to restore service availability. User authentication functionality should return to normal operation. Continue monitoring for service stability.




In [ ]:
# Example 4: External Dependency Failure - Escalation Triggered
# Demonstrates: Dependency analysis → Human escalation
# Scenario: Search index unable to connect to Elasticsearch cluster
run_it_agent("Search functionality is offline. Please investigate search-index-09 service status.")

print("\n" + "="*50 + "\n")


--- New Incident: Search functionality is offline. Please investigate search-index-09 service status. ---

[AI Agent Analyzing Incident...]
-> TOOL: Checking health for search-index-09...
-> TOOL: Fetching last 50 log lines for search-index-09...

[AI Agent Analyzing Incident...]
-> TOOL: Escalating to human...

[AI Agent Analyzing Incident...]

[INCIDENT RESOLUTION]: I have escalated the search-index-09 incident to the engineering team. While the service itself shows healthy resource metrics (CPU 10%, Memory 15%), the system logs indicate a critical infrastructure failure: connection refused to the Elasticsearch cluster at elastic-cluster-main:9200. This indicates an upstream dependency issue that requires specialized infrastructure investigation. The incident has been routed to engineers with full diagnostic context for remediation.




In [ ]:
# Example 5: Preventive Health Verification - Baseline Assessment
# Demonstrates: Routine operational health checks
# Scenario: Frontend service operating within normal parameters
run_it_agent("Conduct routine health assessment of frontend-node-04.")


--- New Incident: Conduct routine health assessment of frontend-node-04. ---

[AI Agent Analyzing Incident...]
-> TOOL: Checking health for frontend-node-04...

[AI Agent Analyzing Incident...]
-> TOOL: Fetching last 50 log lines for frontend-node-04...

[AI Agent Analyzing Incident...]

[INCIDENT RESOLUTION]: Routine health assessment complete for frontend-node-04. All systems operating normally with resource utilization well within acceptable ranges: CPU at 25% and memory at 30%. Application logs show consistent successful request processing with HTTP 200 responses across all endpoints. No errors or warnings detected. System is operating at baseline health status.
